# 03 Classification (DuckDB Input + BLOB Artifacts)

This notebook reads preprocessed session data from DuckDB (READ_ONLY), trains a text classifier, and stores trained artifacts as BLOBs in a DuckDB artifacts database.

In [1]:
from pathlib import Path
import json
import pickle

import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

from utils.duckdb_utils import init_ml_artifacts_table, open_duckdb, query_df
from utils.paths import get_db_paths, resolve_workspace
            


In [2]:
workspace = resolve_workspace(Path.cwd())
input_db, output_db = get_db_paths(workspace)

init_ml_artifacts_table(output_db)

print(f"input_db={input_db}")
print(f"output_db={output_db}")
            


input_db=C:\projects\fab\Fabcon2026\src\notebooks\..\..\Processed\sessions_in_preprocessed.duckdb
output_db=C:\projects\fab\Fabcon2026\src\notebooks\..\..\Processed\sessions_ml_outputs.duckdb


In [3]:
df = query_df(
    input_db,
    '''
    SELECT
      file, title, text, description,
      COALESCE(NULLIF(status, ''), 'Unknown') AS target
    FROM sessions_in_preprocessed
    WHERE COALESCE(NULLIF(text, ''), NULL) IS NOT NULL
    '''
)

class_counts = df['target'].value_counts()
df = df[df['target'].isin(class_counts[class_counts >= 2].index)].copy()

print(df.shape)
print(df['target'].value_counts())
            


(245, 5)
target
Not Reviewed    245
Name: count, dtype: int64


In [4]:
if df['target'].nunique() < 2:
    raise ValueError('Need at least two target classes for classification.')

X_train, X_test, y_train, y_test = train_test_split(
    df['text'], df['target'], test_size=0.25, random_state=42, stratify=df['target']
)

pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=5000, ngram_range=(1, 2), min_df=2)),
    ('clf', LogisticRegression(max_iter=2000, class_weight='balanced'))
])

pipeline.fit(X_train, y_train)
pred = pipeline.predict(X_test)

weighted_f1 = f1_score(y_test, pred, average='weighted')
report = classification_report(y_test, pred, output_dict=True)
metrics = {'weighted_f1': float(weighted_f1), 'report': report}

print(f"weighted_f1={weighted_f1:.4f}")

ValueError: Need at least two target classes for classification.

In [ ]:
pred_df = pd.DataFrame({
    'file': X_test.index.map(df.loc[:, 'file'].to_dict()),
    'true_label': y_test.values,
    'predicted_label': pred
})

with open_duckdb(output_db, read_only=False) as write_con:
    write_con.register('pred_df', pred_df)
    write_con.execute('CREATE OR REPLACE TABLE classification_predictions AS SELECT * FROM pred_df')

    artifact_blob = pickle.dumps(pipeline)
    write_con.execute('''
    INSERT OR REPLACE INTO ml_artifacts
    (artifact_id, notebook, artifact_type, model_name, metrics_json, artifact_blob)
    VALUES (?, ?, ?, ?, ?, ?)
    ''', [
        'classification_logreg_status',
        '03_classification',
        'classification_model',
        'LogisticRegression + TFIDF',
        json.dumps(metrics),
        artifact_blob
    ])

print('Stored classification_predictions table and classification model BLOB artifact.')
            


In [ ]:
query_df(
    output_db,
    '''
    SELECT artifact_id, notebook, artifact_type, model_name, created_at
    FROM ml_artifacts
    WHERE notebook = '03_classification'
    ORDER BY created_at DESC
    '''
)
            
